# StockSight — EDA & Model Backtest

Demand forecasting for cafe inventory. Defensible data science story for interviews.

Run from repo root after `pip install -r requirements.txt`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluate import compare_models, backtest_predictions
from src.ingest import daily_item_sales, load_sales
from src.menu import apply_demo_menu_labels, load_demo_item_map
from src.models import DOW_NAMES

In [ ]:
sales = load_sales(ROOT / "Cleaned_DataSet.csv")
sales = apply_demo_menu_labels(sales, load_demo_item_map())
daily = daily_item_sales(sales)
print(f"Daily rows: {len(daily):,} | items: {daily['item'].nunique()} | {daily['date'].min().date()} to {daily['date'].max().date()}")
daily.head()

## EDA — day-of-week patterns

In [ ]:
dow = daily.copy()
dow["dow_name"] = dow["dow"].map(dict(enumerate(DOW_NAMES)))
by_dow = dow.groupby("dow_name")["units_sold"].mean().reindex(DOW_NAMES)
by_dow.plot(kind="bar", title="Average daily units sold by day of week", figsize=(8, 4))
plt.ylabel("units sold")
plt.tight_layout()
plt.show()

In [ ]:
top = daily.groupby("item")["units_sold"].sum().sort_values(ascending=False).head(8)
top.plot(kind="barh", title="Total units by menu item", figsize=(8, 4))
plt.xlabel("units sold")
plt.tight_layout()
plt.show()

## Model comparison — 28-day holdout backtest

Metrics: **MAE**, **RMSE**, **MAPE**, **stockout rate** (days actual > predicted).

In [ ]:
comparison = compare_models(daily, holdout_days=28, lookback_weeks=8)
comparison

In [ ]:
best_model = comparison.iloc[0]["model"]
preds = backtest_predictions(daily, best_model, holdout_days=28)
preds.groupby("item")[["abs_error"]].mean().sort_values("abs_error", ascending=False).head()

## Interview talking point

Production uses **dow_mean** (day-of-week) for interpretability — managers plan around Saturday lunch rushes.
Re-run backtest on Think Coffee POS data and pick the model that minimizes MAE *and* stockout rate with their buffer policy.